# Multi-Agent RAG with Langfuse Telemetry & Advanced Retrieval

Complete system featuring:
- **Multi-Agent Architecture**: Researcher, Analyzer, Writer agents
- **Advanced Retrieval**: HNSW + MMR + Dense+Sparse Reranking
- **Memory Checkpoints**: LangGraph persistence for multi-turn conversations
- **SQLite Database**: Permanent storage for documents and chat history
- **Langfuse Telemetry**: Full tracing and observability

## 0. Install Dependencies

In [ ]:
# %pip install -U langchain langchain-core langgraph langchain-openai langchain-community langfuse
# %pip install -U hnswlib python-dotenv pydantic sqlalchemy chromadb bm25s rank-bm25
# %pip install -U langchain-text-splitters

## 1. Environment Setup & Langfuse Integration

In [ ]:
import os
from dotenv import load_dotenv
from langfuse import Langfuse
from langfuse.decorators import observe

# Load environment variables
load_dotenv()

# Initialize Langfuse for tracing
# Set these in your .env file:
# LANGFUSE_PUBLIC_KEY=pk_...
# LANGFUSE_SECRET_KEY=sk_...
# LANGFUSE_HOST=https://cloud.langfuse.com

langfuse_client = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
)

print("✓ Langfuse initialized for tracing")

## 2. SQLite Database Setup

In [ ]:
import sqlite3
from datetime import datetime
import json

DB_PATH = "rag_system.db"

def init_database():
    """Initialize SQLite database with required tables."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Documents table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS documents (
        id TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        content TEXT NOT NULL,
        metadata TEXT,
        embedding BLOB,
        sparse_vector TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    
    # Chat history table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chat_history (
        id TEXT PRIMARY KEY,
        session_id TEXT NOT NULL,
        agent_name TEXT NOT NULL,
        role TEXT,
        content TEXT NOT NULL,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        trace_id TEXT
    )
    """)
    
    # Agent state checkpoints
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS agent_checkpoints (
        id TEXT PRIMARY KEY,
        session_id TEXT NOT NULL,
        agent_name TEXT NOT NULL,
        state TEXT NOT NULL,
        step_number INTEGER,
        timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)
    
    conn.commit()
    conn.close()
    print("✓ SQLite database initialized at", DB_PATH)

init_database()

## 3. Advanced Retrieval Setup (HNSW + MMR + Reranking)

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.docstore.in_memory import InMemoryDocstore
import hnswlib
from typing import List, Tuple
import uuid

# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

class AdvancedRetriever:
    """Retriever with HNSW + MMR + Dense+Sparse Reranking."""
    
    def __init__(self, embedding_dim: int = 1536):
        self.embedding_dim = embedding_dim
        self.hnsw_index = hnswlib.Index(space='cosine', dim=embedding_dim)
        self.hnsw_index.init_index(max_elements=10000, ef_construction=200, M=64)
        self.documents = {}  # Store docs by ID
        self.doc_embeddings = []  # Store embeddings
        self.next_id = 0
        
    @observe()
    def add_documents(self, docs: List[dict]):
        """Add documents with embeddings to HNSW index."""
        for doc in docs:
            doc_id = str(uuid.uuid4())
            embedding = np.array(embeddings.embed_query(doc["content"]), dtype=np.float32)
            self.hnsw_index.add_items(embedding.reshape(1, -1), [self.next_id])
            self.documents[self.next_id] = {"id": doc_id, **doc}
            self.doc_embeddings.append(embedding)
            self.next_id += 1
    
    @observe()
    def retrieve_hnsw(self, query: str, k: int = 10) -> List[dict]:
        """Retrieve using HNSW (Hierarchical Navigable Small World)."""
        query_embedding = np.array(embeddings.embed_query(query), dtype=np.float32)
        labels, distances = self.hnsw_index.knn_query(query_embedding.reshape(1, -1), k=k)
        results = [self.documents[idx] for idx in labels[0] if idx in self.documents]
        return results
    
    @observe()
    def retrieve_mmr(self, query: str, k: int = 5, lambda_mult: float = 0.5) -> List[dict]:
        """Maximal Marginal Relevance (MMR) retrieval for diversity."""
        query_embedding = np.array(embeddings.embed_query(query), dtype=np.float32)
        
        # Get initial candidates
        candidates_labels, _ = self.hnsw_index.knn_query(query_embedding.reshape(1, -1), k=min(20, len(self.documents)))
        candidates = [self.documents[idx] for idx in candidates_labels[0] if idx in self.documents]
        
        # MMR selection
        selected = []
        candidate_indices = set(range(len(candidates)))
        
        while len(selected) < k and candidate_indices:
            best_idx = None
            best_score = -float('inf')
            
            for idx in candidate_indices:
                candidate_emb = np.array(embeddings.embed_query(candidates[idx]["content"]), dtype=np.float32)
                relevance = np.dot(query_embedding, candidate_emb)
                
                if selected:
                    selected_embs = np.array([np.array(embeddings.embed_query(candidates[s]["content"]), dtype=np.float32) for s in selected])
                    diversity = np.min([np.dot(candidate_emb, se) for se in selected_embs])
                else:
                    diversity = 1.0
                
                score = lambda_mult * relevance - (1 - lambda_mult) * diversity
                if score > best_score:
                    best_score = score
                    best_idx = idx
            
            if best_idx is not None:
                selected.append(best_idx)
                candidate_indices.remove(best_idx)
        
        return [candidates[i] for i in selected]

print("✓ Advanced Retriever initialized with HNSW + MMR")

## 4. Sparse & Dense Reranking

In [ ]:
from rank_bm25 import BM25Okapi
import re

class DenseSpaceReranker:
    """Reranker combining dense (semantic) and sparse (lexical) signals."""
    
    def __init__(self):
        self.embeddings = embeddings
        self.bm25 = None
        self.corpus = []
    
    def fit(self, documents: List[dict]):
        """Fit BM25 on corpus."""
        self.corpus = documents
        tokenized_corpus = [self._tokenize(doc["content"]) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization."""
        return re.findall(r'\w+', text.lower())
    
    @observe()
    def rerank(self, query: str, candidates: List[dict], k: int = 5, 
               dense_weight: float = 0.7, sparse_weight: float = 0.3) -> List[dict]:
        """Rerank using combined dense+sparse scores."""
        query_embedding = np.array(self.embeddings.embed_query(query), dtype=np.float32)
        
        scores = []
        for doc in candidates:
            # Dense score
            doc_embedding = np.array(self.embeddings.embed_query(doc["content"]), dtype=np.float32)
            dense_score = np.dot(query_embedding, doc_embedding)
            
            # Sparse score (BM25)
            query_tokens = self._tokenize(query)
            sparse_scores = self.bm25.get_scores(query_tokens) if self.bm25 else [0] * len(self.corpus)
            sparse_score = sparse_scores[self.corpus.index(doc)] if doc in self.corpus else 0
            
            # Combined score
            combined_score = dense_weight * dense_score + sparse_weight * (sparse_score / 100.0)
            scores.append((doc, combined_score))
        
        # Sort by combined score
        scores.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scores[:k]]

print("✓ Dense+Sparse Reranker initialized")

## 5. Multi-Agent Architecture with Memory Checkpoints

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END, START
from typing import Annotated, TypedDict, Literal
import operator

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

class AgentState(TypedDict):
    """Shared state for multi-agent system."""
    task: str
    context: str
    retrieved_docs: Annotated[list, operator.add]  # Append-only
    researcher_output: str
    analyzer_output: str
    writer_output: str
    session_id: str
    trace_id: str
    messages: Annotated[list, operator.add]  # Chat history

# Agent prompts
researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research specialist. Given a task, search for and retrieve relevant information."),
    ("user", "Task: {task}\n\nContext: {context}")
])

analyzer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an analysis expert. Analyze retrieved information critically."),
    ("user", "Task: {task}\n\nResearch: {researcher_output}")
])

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Create clear, structured output."),
    ("user", "Task: {task}\n\nAnalysis: {analyzer_output}")
])

@observe()
def researcher_node(state: AgentState) -> AgentState:
    """Research agent: retrieves and summarizes information."""
    output = "Research findings: [placeholder]"
    return {"researcher_output": output, "messages": [{"role": "researcher", "content": output}]}

@observe()
def analyzer_node(state: AgentState) -> AgentState:
    """Analyzer agent: critiques and analyzes research."""
    output = "Analysis: [placeholder]"
    return {"analyzer_output": output, "messages": [{"role": "analyzer", "content": output}]}

@observe()
def writer_node(state: AgentState) -> AgentState:
    """Writer agent: produces final output."""
    output = "Final output: [placeholder]"
    return {"writer_output": output, "messages": [{"role": "writer", "content": output}]}

print("✓ Multi-agent nodes initialized")

## 6. Build LangGraph with Checkpoints

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# Create checkpoint saver for memory persistence
checkpoint_storage = SqliteSaver.from_conn_string(f"sqlite:///{DB_PATH}")

# Build graph
builder = StateGraph(AgentState)
builder.add_node("researcher", researcher_node)
builder.add_node("analyzer", analyzer_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_edge("researcher", "analyzer")
builder.add_edge("analyzer", "writer")
builder.add_edge("writer", END)

# Compile with checkpoint storage for memory persistence
agent_graph = builder.compile(checkpointer=checkpoint_storage)

print("✓ LangGraph compiled with SQLite checkpoint storage")

## 7. Logging & Tracing Utilities with Langfuse

In [ ]:
import logging
from uuid import uuid4

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class TelemetryLogger:
    """Logs to both SQLite and Langfuse."""
    
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.trace_id = str(uuid4())
        self.langfuse_client = langfuse_client
    
    @observe()
    def log_agent_message(self, agent_name: str, role: str, content: str):
        """Log agent message to SQLite and Langfuse."""
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        
        msg_id = str(uuid4())
        cursor.execute("""
            INSERT INTO chat_history (id, session_id, agent_name, role, content, trace_id)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (msg_id, self.session_id, agent_name, role, content, self.trace_id))
        conn.commit()
        conn.close()
        
        # Log to Langfuse
        self.langfuse_client.log_message(
            trace_id=self.trace_id,
            message_id=msg_id,
            role=role,
            content=content,
            session_id=self.session_id
        )
        
        logger.info(f"[{agent_name}] {role}: {content[:100]}...")
    
    def log_checkpoint(self, agent_name: str, step: int, state: dict):
        """Save agent state checkpoint."""
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        
        checkpoint_id = str(uuid4())
        state_json = json.dumps(state, default=str)
        
        cursor.execute("""
            INSERT INTO agent_checkpoints (id, session_id, agent_name, state, step_number)
            VALUES (?, ?, ?, ?, ?)
        """, (checkpoint_id, self.session_id, agent_name, state_json, step))
        conn.commit()
        conn.close()
        
        logger.info(f"Checkpoint saved: {agent_name} step {step}")

print("✓ Telemetry logger with Langfuse integration initialized")

## 8. Sample Usage: Run Multi-Agent RAG

In [ ]:
# Create retriever, reranker, and telemetry
retriever = AdvancedRetriever()
reranker = DenseSpaceReranker()

# Sample documents
sample_docs = [
    {"title": "RAG Basics", "content": "Retrieval-Augmented Generation combines retrieval and generation models..."},
    {"title": "Vector DBs", "content": "Vector databases store embeddings for semantic search..."},
    {"title": "Multi-Agent", "content": "Multi-agent systems coordinate multiple specialized agents..."},
]

# Add documents to retriever
retriever.add_documents(sample_docs)
reranker.fit(sample_docs)

# Create telemetry logger
session_id = str(uuid4())
telemetry = TelemetryLogger(session_id)

print(f"\n✓ Session started: {session_id}")
print(f"✓ Trace ID: {telemetry.trace_id}")

## 9. Execute Multi-Agent Workflow

In [ ]:
@observe()
def run_rag_pipeline(task: str, session_id: str):
    """Execute complete RAG pipeline with all agents and tracing."""
    telemetry = TelemetryLogger(session_id)
    
    # Step 1: Retrieve documents (HNSW)
    retrieved = retriever.retrieve_hnsw(task, k=10)
    telemetry.log_agent_message("retriever", "system", f"Retrieved {len(retrieved)} documents")
    
    # Step 2: Apply MMR for diversity
    diverse_docs = retriever.retrieve_mmr(task, k=5, lambda_mult=0.5)
    telemetry.log_agent_message("retriever", "system", f"Applied MMR, selected {len(diverse_docs)} diverse results")
    
    # Step 3: Rerank with dense+sparse
    reranked = reranker.rerank(task, retrieved, k=3, dense_weight=0.7, sparse_weight=0.3)
    telemetry.log_agent_message("reranker", "system", f"Reranked to top {len(reranked)} results")
    
    # Step 4: Run multi-agent workflow
    initial_state = {
        "task": task,
        "context": "\n".join([f"- {doc['title']}: {doc['content'][:150]}" for doc in reranked]),
        "retrieved_docs": reranked,
        "researcher_output": "",
        "analyzer_output": "",
        "writer_output": "",
        "session_id": session_id,
        "trace_id": telemetry.trace_id,
        "messages": []
    }
    
    result = agent_graph.invoke(
        initial_state,
        config={"configurable": {"thread_id": session_id}}
    )
    
    # Log checkpoint
    telemetry.log_checkpoint("workflow", 1, result)
    
    return result

# Run pipeline
task = "Explain multi-agent RAG systems"
result = run_rag_pipeline(task, session_id)

print("\n" + "="*60)
print("MULTIAGENT RAG EXECUTION COMPLETE")
print("="*60)
print(f"\nSession ID: {session_id}")
print(f"Trace ID: {result['trace_id']}")
print(f"\nWriter Output:\n{result.get('writer_output', 'N/A')}")

## 10. View Telemetry & Chat History

In [ ]:
@observe()
def view_session_history(session_id: str):
    """Retrieve and display chat history for a session."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT agent_name, role, content, timestamp FROM chat_history
        WHERE session_id = ? ORDER BY timestamp
    """, (session_id,))
    
    messages = cursor.fetchall()
    conn.close()
    
    print(f"\n📜 Session History: {session_id}\n")
    for msg in messages:
        print(f"[{msg['agent_name']}] {msg['role']}: {msg['content'][:100]}...")
        print(f"   Timestamp: {msg['timestamp']}\n")

@observe()
def view_checkpoints(session_id: str):
    """Retrieve and display checkpoints for a session."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT agent_name, step_number, timestamp FROM agent_checkpoints
        WHERE session_id = ? ORDER BY step_number
    """, (session_id,))
    
    checkpoints = cursor.fetchall()
    conn.close()
    
    print(f"\n🔖 Checkpoints: {session_id}\n")
    for cp in checkpoints:
        print(f"Step {cp['step_number']}: {cp['agent_name']} @ {cp['timestamp']}")

# View history and checkpoints
view_session_history(session_id)
view_checkpoints(session_id)

print("\n✓ Access Langfuse dashboard: https://cloud.langfuse.com")
print(f"✓ Look for trace ID: {telemetry.trace_id}")

## 11. Summary

This notebook demonstrates:

1. **Multi-Agent Architecture**: Researcher → Analyzer → Writer pipeline
2. **Advanced Retrieval**: 
   - HNSW for efficient similarity search
   - MMR for diverse results
   - Dense+Sparse reranking (semantic + lexical)
3. **Memory Checkpoints**: LangGraph with SqliteSaver for state persistence
4. **SQLite Database**: Permanent storage for documents, chat history, and checkpoints
5. **Langfuse Telemetry**: Full tracing and observability

**Next Steps**:
- Configure your Langfuse credentials in `.env`
- Scale retriever to real documents
- Implement actual LLM calls in agent nodes
- Monitor traces in Langfuse dashboard